In [ ]:
# 1. Cai dat thu vien can thiet (neu thieu)
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "gymnasium", "pyyaml"],
    check=True,
)

print("Da cai gymnasium + pyyaml (neu truoc do chua co).")


## 2. Clone GitHub repo vao `/kaggle/working/repo`

Notebook nay dung Kaggle Secret `GH_TOKEN` de clone/pull repo truoc khi train Branching DDQ.

In [ ]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import subprocess
import sys

GITHUB_REPO = "sin0235/Project"
BRANCH = "master"

user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GH_TOKEN")

CLONE_DIR = Path("/kaggle/working/repo")

if CLONE_DIR.exists():
    result = subprocess.run(
        ["git", "-C", str(CLONE_DIR), "pull"],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
else:
    repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    result = subprocess.run(
        ["git", "clone", "--depth=1", "-b", BRANCH, repo_url, str(CLONE_DIR)],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

print(result.stdout)
if result.returncode != 0:
    raise RuntimeError("Git clone/pull that bai. Kiem tra lai TOKEN, REPO va BRANCH.")

PROJECT_ROOT = CLONE_DIR
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")


## 3. Doc cau hinh Branching DDQ

Notebook uu tien `Conf/branching_ddq_conf.yaml`. Neu file nay chua co thi fallback sang `Conf/ddq_conf.yaml`, va cuoi cung dung `DEFAULT_CONFIG` trong `src/training/BranchingDDQ.py`.

In [ ]:
import yaml
from pprint import pprint

from src.training.BranchingDDQ import resolve_branching_ddq_config

candidate_configs = [
    PROJECT_ROOT / "Conf" / "branching_ddq_conf.yaml",
    PROJECT_ROOT / "Conf" / "ddq_conf.yaml",
]

selected_conf = None
raw_cfg = {}

for conf_path in candidate_configs:
    if not conf_path.exists():
        continue
    with open(conf_path, "r", encoding="utf-8") as f:
        loaded_cfg = yaml.safe_load(f) or {}
    if not isinstance(loaded_cfg, dict):
        raise ValueError(f"Config phai la dict: {conf_path}")
    selected_conf = conf_path
    raw_cfg = loaded_cfg
    if loaded_cfg:
        break

branching_cfg = resolve_branching_ddq_config(config=raw_cfg)

# Override duong dan data khi chay tren Kaggle
branching_cfg["data_path"] = "/kaggle/input/datasets/phctontrn/dataset-trading-rl"

if raw_cfg:
    config_source_label = str(selected_conf)
elif selected_conf is not None:
    config_source_label = f"{selected_conf} (empty, fallback DEFAULT_CONFIG)"
else:
    config_source_label = "DEFAULT_CONFIG in src/training/BranchingDDQ.py"

print("Config source:", config_source_label)
print("Data path (Kaggle):", branching_cfg["data_path"])
print("\nBranching DDQ config (rut gon):")
keys_to_show = [
    "tickers", "features", "window_size",
    "train_ratio", "val_ratio", "test_ratio",
    "initial_balance", "fee_rate", "reward_name", "reward_window",
    "hidden_size", "num_layers", "dropout", "k",
    "learning_rate", "gamma", "tau", "batch_size",
    "replay_buffer_size", "learning_starts", "train_freq", "gradient_steps",
    "epsilon_start", "epsilon_end", "epsilon_decay_steps",
    "total_timesteps", "eval_freq", "save_freq", "n_eval_episodes",
]
summary = {k: branching_cfg.get(k) for k in keys_to_show}
pprint(summary)


## 4. Chay training Branching DDQ + LSTM

Cell duoi se goi truc tiep `train_branching_ddq(config)` trong `src/training/BranchingDDQ.py`.

- Logger se tao `run_id` va ghi vao `results/runs/<run_id>`.
- Branching DDQ tra action vector theo tung ma thay vi 1 scalar action.
- Co the override nhanh hyperparameter ngay trong notebook truoc khi train.

In [ ]:
import os
import sys

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print("CWD:", os.getcwd())

from src.training.BranchingDDQ import train_branching_ddq

# Co the override them mot so hyperparameters neu muon, vi du:
# branching_cfg["total_timesteps"] = 200_000
# branching_cfg["learning_rate"] = 2e-4
# branching_cfg["eval_freq"] = 5_000

agent, test_metrics = train_branching_ddq(branching_cfg)

print("\nKet qua test trung binh (tom tat):")
for k, v in sorted(test_metrics.items()):
    print(f"  {k}: {v}")


## 5. Eval chi tiet va xem lai ket qua

Cell duoi tu tim run Branching DDQ moi nhat con dung duoc, uu tien `best_model.pt`, neu khong co thi fallback sang `final_model.pt` hoac checkpoint gan nhat.

In [ ]:
import sys

sys.path.insert(0, str(PROJECT_ROOT))

from src.training.BranchingDDQ import (
    evaluate_branching_ddq,
    resolve_branching_ddq_config,
    resolve_branching_eval_run_across_roots,
)
from src.training.DDQ import get_results_root_candidates

base_cfg = resolve_branching_ddq_config(config=branching_cfg)
base_cfg["data_path"] = "/kaggle/input/datasets/phctontrn/dataset-trading-rl"

candidate_roots = get_results_root_candidates(project_root=PROJECT_ROOT)
resolved = resolve_branching_eval_run_across_roots(
    candidate_roots,
    base_config=base_cfg,
    overrides={
        "data_path": "/kaggle/input/datasets/phctontrn/dataset-trading-rl",
        "device": "auto",
    },
)

if resolved["run_dir"] is None:
    print("Khong tim thay run Branching DDQ nao du artifact de eval.")
    if resolved["checked_results_roots"]:
        print("Da kiem tra cac thu muc:")
        for root in resolved["checked_results_roots"]:
            print(" -", root)
    if resolved["missing_results_roots"]:
        print("Thieu thu muc:")
        for root in resolved["missing_results_roots"]:
            print(" -", root)
    if resolved["all_skipped_runs"]:
        print("Cac run bi bo qua:")
        for item in resolved["all_skipped_runs"]:
            print(" -", item["run_id"], "->", item["reason"], "| root:", item["results_root"])
else:
    print("Results root:", resolved["results_root"])
    print("Latest evaluable run:", resolved["run_dir"].name)
    print("Config source:", resolved["config_source"])
    print("Checkpoint source:", resolved["ckpt_source"])
    if resolved["all_skipped_runs"]:
        print("Skipped runs during search:")
        for item in resolved["all_skipped_runs"]:
            print(" -", item["run_id"], "->", item["reason"], "| root:", item["results_root"])

    eval_result = evaluate_branching_ddq(
        config={
            "data_path": "/kaggle/input/datasets/phctontrn/dataset-trading-rl",
            "device": "auto",
        },
        run_dir=str(resolved["run_dir"]),
    )

    print("\nMetrics dict:")
    for k, v in sorted(eval_result["metrics"].items()):
        print(f"  {k}: {v}")


In [ ]:
import shutil
import sys

sys.path.insert(0, str(PROJECT_ROOT))

from src.training.DDQ import get_results_root_candidates

candidate_roots = get_results_root_candidates(project_root=PROJECT_ROOT)
runs_dir = next((root for root in candidate_roots if root.exists() and any(root.iterdir())), None)

if runs_dir is not None:
    zip_base = runs_dir.parent / "runs_backup_branching_ddq"
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=runs_dir)
    print(f"Da tao file zip: {zip_path}")
    print(f"Source runs dir: {runs_dir}")
else:
    print("Khong tim thay thu muc runs nao co du lieu.")
    print("Da kiem tra:")
    for root in candidate_roots:
        print(" -", root)
